# 1.14 下一字怎樣接成一段新文字？


給一個開頭「貓」，希望接出後續文字。這時沒有真實答案可對照：先算下一字分配、選一字、接回句尾，再把新字當下一次前文。重複接字就是**生成**。

為了只觀察這個流程，我們手動指定四字機率表，沒有訓練。字表為 `['貓','看','狗','。']`，ID 為 0、1、2、3。看到貓時取第 0 列，看的機率為 0.85；若選到看，下次就取第 1 列，而不是繼續用貓那列。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

chars = ["貓", "看", "狗", "。"]
p = torch.tensor(
    [
        [0.05, 0.85, 0.05, 0.05],
        [0.40, 0.05, 0.50, 0.05],
        [0.05, 0.70, 0.05, 0.20],
        [0.50, 0.05, 0.40, 0.05],
    ]
)
g = torch.Generator().manual_seed(42)
current = 0
output = [chars[current]]
for _ in range(12):
    current = torch.multinomial(p[current], 1, generator=g).item()
    output.append(chars[current])
print("".join(output))
assert len(output) == 13

`torch.multinomial(p[current],1)` 按目前列的比例抽一個 ID。`.item()` 取出整數，`append` 把對應字加到文字尾端；新 ID 也成為下一輪的 `current`。開頭已有一字，再抽十二次，結果共十三字。

表裡低機率組合仍可能被抽中，因此可出現「貓看狗」，也可出現連續貓或句號。這段指定抽十二次才結束，句號只是普通候選，沒有自動停止功能。後面做對話時會另教結束標記。

生成也能暴露只看一字的限制。現有接字表實驗用 12 篇「顏色=紅；形狀=圓。」這類短文，拿 9 篇訓練。給開頭 `顏色=`，每步選最高分，它接出 `三角。`，合起來是「顏色=三角。」：三角是形狀，填錯欄位。

原因不是少算一次：顏色欄與形狀欄都以等號結尾，只看最近一字的表收到相同輸入，自然給相同分配。訓練代價下降仍可能伴隨這種生成錯誤。下一章將更多前文送入模型，讓它有機會分清兩個欄位。

練習把種子 42 改成 7，字表、機率、開頭和總長度都不變，抽到的文字可能改變。再把十二次改成三次，斷言長度也改成四，核對每輪只新增一字。

<details>
<summary>補充與重做</summary>

`torch.Generator().manual_seed(42)` 固定本段抽樣工具的起點，通常能在相同軟體與裝置重現序列；不保證跨版本完全一樣。`range(12)` 重複十二次，`_` 表示不用那個迴圈編號。接字表的完整配方與輸出見 [T.3](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.3)。

</details>
